# Exercise: Neural Networks

## 1. Import the required libraries in the cell below

In [1]:
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error
from keras.models import Sequential
from keras.layers import Dense, Input
from keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt

## 2. Read the data

In [2]:
housing = pd.read_csv('data/sacramento_real_estate_transactions.csv')
housing.head()


,street,city,zip,state,beds,baths,sq__ft,type,sale_date,price,latitude,longitude
0,3526 HIGH ST,SACRAMENTO,95838,CA,2,1,836,Residential,Wed May 21 00:00:00 EDT 2008,59222,38.631913,-121.434879
1,51 OMAHA CT,SACRAMENTO,95823,CA,3,1,1167,Residential,Wed May 21 00:00:00 EDT 2008,68212,38.478902,-121.431028
2,2796 BRANCH ST,SACRAMENTO,95815,CA,2,1,796,Residential,Wed May 21 00:00:00 EDT 2008,68880,38.618305,-121.443839
3,2805 JANETTE WAY,SACRAMENTO,95815,CA,2,1,852,Residential,Wed May 21 00:00:00 EDT 2008,69307,38.616835,-121.439146
4,6001 MCMAHON DR,SACRAMENTO,95824,CA,2,1,797,Residential,Wed May 21 00:00:00 EDT 2008,81900,38.519470,-121.435768


## 3. Conduct Exploratory Data Analysis 

In [3]:
# Check data info
housing.info()

<class 'pandas.DataFrame'>
RangeIndex: 985 entries, 0 to 984
Data columns (total 12 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   street     985 non-null    str    
 1   city       985 non-null    str    
 2   zip        985 non-null    int64  
 3   state      985 non-null    str    
 4   beds       985 non-null    int64  
 5   baths      985 non-null    int64  
 6   sq__ft     985 non-null    int64  
 7   type       985 non-null    str    
 8   sale_date  985 non-null    str    
 9   price      985 non-null    int64  
 10  latitude   985 non-null    float64
 11  longitude  985 non-null    float64
dtypes: float64(2), int64(5), str(5)
memory usage: 92.5 KB


In [4]:
# Check for missing values
housing.isna().sum()

street       0
city         0
zip          0
state        0
beds         0
baths        0
sq__ft       0
type         0
sale_date    0
price        0
latitude     0
longitude    0
dtype: int64

In [5]:
# Check data types
housing.dtypes

street           str
city             str
zip            int64
state            str
beds           int64
baths          int64
sq__ft         int64
type             str
sale_date        str
price          int64
latitude     float64
longitude    float64
dtype: object

In [6]:
# Look at the Summary Stats.
housing.describe()

,zip,beds,baths,sq__ft,price,latitude,longitude
count,985.000000,985.000000,985.000000,985.000000,985.000000,985.000000,985.000000
mean,95750.697462,2.911675,1.776650,1312.918782,233715.951269,38.445121,-121.193371
std,85.176072,1.307932,0.895371,856.123224,139088.818896,5.103637,5.100670
min,95603.000000,0.000000,0.000000,-984.000000,-210944.000000,-121.503471,-121.551704
25%,95660.000000,2.000000,1.000000,950.000000,145000.000000,38.482704,-121.446119
50%,95762.000000,3.000000,2.000000,1304.000000,213750.000000,38.625932,-121.375799
75%,95828.000000,4.000000,2.000000,1718.000000,300000.000000,38.695589,-121.294893
max,95864.000000,8.000000,5.000000,5822.000000,884790.000000,39.020808,38.668433


In [7]:
# Cleaning Steps (if needed)
# There is one row with a negative price and swapped lat/long values -
# this is clearly a bad record, so we drop it.
housing = housing[housing['price'] > 0]

# Drop the single row with an 'Unkown' (typo) property type - not usable.
housing = housing[housing['type'] != 'Unkown']

housing.shape


(983, 12)

In [8]:
# Rows with sq__ft == 0 don't have real square footage recorded
# (mostly vacant-lot style listings) - drop them since sq__ft is a key feature.
housing = housing[housing['sq__ft'] > 0]
housing.shape


(813, 12)

In [9]:
# Reset the index after dropping rows
housing = housing.reset_index(drop=True)
housing.describe()


,zip,beds,baths,sq__ft,price,latitude,longitude
count,813.000000,813.000000,813.000000,813.000000,813.000000,813.000000,813.000000
mean,95761.400984,3.247232,1.961870,1591.892989,229471.130381,38.576931,-121.378533
std,85.357516,0.849012,0.669367,663.908347,119897.576889,0.126352,0.119730
min,95603.000000,1.000000,1.000000,484.000000,2000.000000,38.241514,-121.550527
25%,95670.000000,3.000000,2.000000,1144.000000,148750.000000,38.473814,-121.451444
50%,95820.000000,3.000000,2.000000,1419.000000,207973.000000,38.591618,-121.404999
75%,95828.000000,4.000000,2.000000,1851.000000,285000.000000,38.674864,-121.325730
max,95864.000000,8.000000,5.000000,5822.000000,884790.000000,39.008159,-120.597599


In [10]:
#change sale_date types


## 4. Split the Data & Pre-Processing
**Best Practice for Neural Network** Split your data before modeling into three part:

1. __The Training Set (The Textbook):__ This is the data the neural network actually looks at to learn. During the training loop, the network uses this data to adjust its **weights** and **biases** via backpropagation.

2. __The Validation Set (The Practice Quiz):__ This data evaluates the model *during* training to fine-tune settings. The model does not learn its weights from this data. Instead, the it uses the validation score to choose **hyperparameters** (like learning rate, number of layers, or node count) and to decide when to stop training (**early stopping**).


3. __The Test Set (The Final Exam):__ This data provides an unbiased, final evaluation of the model's real-world performance. This dataset is locked away until training is completely finished. It is only used *once* at the very end to see how well the network performs on truly unseen data.

__Typical Split Ratios__

Depending on the size of your total dataset, data scientists generally use these proportions:

| Dataset Size | Train | Validation | Test |
| :--- | :--- | :--- | :--- |
| **Traditional (Small/Medium Data)** | 60% | 20% | 20% |
| **Modern Deep Learning (Millions of Rows)** | 98% | 1% | 1% |

In [11]:
# Select the Features and Target
X = housing[['sq__ft']]
y = housing['price']


In [12]:
# Split to Train Test & Validation
# First split off the Test set (the "Final Exam")
X_train_all, X_test, y_train_all, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42)

# Then split the remaining data into Train and Validation (the "Practice Quiz")
X_train, X_val, y_train, y_val = train_test_split(
    X_train_all, y_train_all, test_size=0.20, random_state=42)

print(X_train.shape, X_val.shape, X_test.shape)


(520, 1) (130, 1) (163, 1)


In [13]:
# IMPORTANT: Scale the features for Neural Networks
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Also scale the "all training" set (train + val) since Linear Regression
# and later models are trained on it
X_train_all_scaled = scaler.transform(X_train_all)


## 5. Linear Regression

In [14]:
# Initialize LinearRegression model
lr = LinearRegression()


In [15]:
# Fit and check the Training Score
lr.fit(X_train_all, y_train_all)
lr.score(X_train_all, y_train_all)


0.49925583857152667

In [16]:
# Predict and check the Testing Score
y_pred_lr = lr.predict(X_test)
lr.score(X_test, y_test)


0.39788582018348173

In [17]:
# Evaluate the model
# Coefficient
print('Coefficient:', lr.coef_)

# Intercept
print('Intercept:', lr.intercept_)

# RMSE
print('RMSE:', root_mean_squared_error(y_test, y_pred_lr))


Coefficient: [125.62893564]
Intercept: 29497.568792874372
RMSE: 87462.31374512309


## 6. Build a Single Neural Network
Creating a model in `keras` entails a few steps:
1. Create your network topology
2. Compile your model
3. Fit your model
   
**Important Note on Regression:**
Because we are predicting a continuous value (house price), our network is performing regression, not classification. 
* Your **hidden layers** should generally use the `relu` activation function.
* Your **output layer** must have exactly **1 neuron** and **no activation function** (or `activation='linear'`) so it can predict any continuous number.
* When compiling, use `loss='mse'` (Mean Squared Error) and `metrics=['mae']` (Mean Absolute Error) so we can see the error in actual dollar amounts.

In [18]:
# Step 1: Define
model = Sequential()
model.add(Input(shape=(1,)))            # 1 feature (sq__ft) coming in
model.add(Dense(1, activation='linear'))  # output layer: 1 neuron, no/linear activation

# Step 2: Compile
model.compile(loss='mse', metrics=['mae'])

# Step 3: Fit
model.fit(X_train_scaled, y_train,
          validation_data=(X_val_scaled, y_val))


17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 68708024320.0000 - mae: 233264.8281 - val_loss: 65196670976.0000 - val_mae: 220649.6250


In [19]:
# Model Weights
model.weights


[<Variable path=sequential/dense/kernel, shape=(1, 1), dtype=float32, value=[[1.5138211]]>,
 <Variable path=sequential/dense/bias, shape=(1,), dtype=float32, value=[0.02549207]>]

In [20]:
# Model Summary
model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense (Dense)                        │ (None, 1)                   │               2 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 6 (28.00 B)

 Trainable params: 2 (8.00 B)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 4 (20.00 B)

## 7. Stacking Multiple Neurons

#### Shallow NN

In [21]:
# Step 1: Define
model = Sequential()
model.add(Input(shape=(1,)))
model.add(Dense(3))              # hidden layer with 3 neurons
model.add(Dense(1))              # output layer

# Step 2: Compile
model.compile(loss='mse', metrics=['mae'])

# Step 3: Fit
model.fit(X_train_scaled, y_train,
          validation_data=(X_val_scaled, y_val))


17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 68708458496.0000 - mae: 233264.7969 - val_loss: 65196961792.0000 - val_mae: 220649.2969


In [22]:
# Model Weights
model.weights


[<Variable path=sequential_1/dense_1/kernel, shape=(1, 3), dtype=float32, value=[[0.222762  0.6080705 1.0738325]]>,
 <Variable path=sequential_1/dense_1/bias, shape=(3,), dtype=float32, value=[ 0.02501933 -0.02459144 -0.02484054]>,
 <Variable path=sequential_1/dense_2/kernel, shape=(3, 1), dtype=float32, value=[[ 0.9680462 ]
  [-0.24427333]
  [-1.0210838 ]]>,
 <Variable path=sequential_1/dense_2/bias, shape=(1,), dtype=float32, value=[0.02492294]>]

In [23]:
# Model Summary
model.summary()


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense_1 (Dense)                      │ (None, 3)                   │               6 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 1)                   │               4 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 22 (92.00 B)

 Trainable params: 10 (40.00 B)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 12 (52.00 B)

#### Deep Network

In [24]:
# use multiple hidden layers with different neurons

# Step 1: Define
model = Sequential()
model.add(Input(shape=(1,)))
model.add(Dense(30))
model.add(Dense(20))
model.add(Dense(15))
model.add(Dense(1))              # output layer

# Step 2: Compile
model.compile(loss='mse', metrics=['mae'])

# Step 3: Fit
model.fit(X_train_scaled, y_train,
          validation_data=(X_val_scaled, y_val))


17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - loss: 68708126720.0000 - mae: 233264.5938 - val_loss: 65196511232.0000 - val_mae: 220649.0156


In [25]:
# Model Weights
model.weights


[<Variable path=sequential_2/dense_3/kernel, shape=(1, 30), dtype=float32, value=[[ 0.34810877  0.08372881 -0.04958724  0.08115898 -0.24507831  0.11633919
    0.18536976 -0.2319394   0.3917471  -0.18120344 -0.2902971   0.1102142
   -0.2229796  -0.17851563 -0.09223852 -0.21854763  0.13223274 -0.00553826
   -0.2462144  -0.24655701 -0.2958319   0.2984706  -0.21079025 -0.21232778
   -0.19136946 -0.13635483  0.4240632  -0.2662403  -0.2307059  -0.0618221 ]]>,
 <Variable path=sequential_2/dense_3/bias, shape=(30,), dtype=float32, value=[ 0.02897051 -0.02552911 -0.02685786  0.02701112 -0.02708175  0.00297697
   0.00711433 -0.02746197 -0.00575178 -0.02816702 -0.01175436  0.02957314
  -0.02794633 -0.02879754  0.02404857  0.02403044 -0.02548232 -0.02667475
  -0.02430701 -0.028396    0.0253428   0.03128735  0.02823014  0.02140348
  -0.02806418  0.0204159   0.0304393  -0.02725031  0.01637981 -0.02610265]>,
 <Variable path=sequential_2/dense_4/kernel, shape=(30, 20), dtype=float32, value=[[-5.844656

In [26]:
# Model Summary
model.summary()


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                      │ (None, 30)                  │              60 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_4 (Dense)                      │ (None, 20)                  │             620 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_5 (Dense)                      │ (None, 15)                  │             315 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_6 (Dense)                      │ (None, 1)                   │              16 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 2,024 (7.91 KB)

 Trainable params: 1,011 (3.95 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 1,013 (3.96 KB)

## 8. Activation Functions

In [27]:
# from keras.layers import Activation
# Note: import Activation if you use it as a standalone layer

# Step 1: Define
model = Sequential()
model.add(Input(shape=(1,)))
model.add(Dense(3, activation='relu'))     # hidden layer with relu activation
model.add(Dense(1, activation='linear'))   # output layer with linear activation

# Step 2: Compile
model.compile(loss='mse', metrics=['mae'])

# Step 3: Fit
model.fit(X_train_scaled, y_train,
          validation_data=(X_val_scaled, y_val))


17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 68708323328.0000 - mae: 233264.9531 - val_loss: 65196883968.0000 - val_mae: 220649.5156


In [28]:
# Model Weights
model.weights


[<Variable path=sequential_3/dense_7/kernel, shape=(1, 3), dtype=float32, value=[[-0.31815216  0.76532483 -0.534513  ]]>,
 <Variable path=sequential_3/dense_7/bias, shape=(3,), dtype=float32, value=[ 0.02657606 -0.02176572 -0.02528455]>,
 <Variable path=sequential_3/dense_8/kernel, shape=(3, 1), dtype=float32, value=[[ 0.91992676]
  [-0.06020916]
  [-0.8753258 ]]>,
 <Variable path=sequential_3/dense_8/bias, shape=(1,), dtype=float32, value=[0.02485139]>]

In [29]:
# Model Summary
model.summary()


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense_7 (Dense)                      │ (None, 3)                   │               6 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_8 (Dense)                      │ (None, 1)                   │               4 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 22 (92.00 B)

 Trainable params: 10 (40.00 B)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 12 (52.00 B)

## 9. Epochs

In [30]:
# Step 1: Define
model = Sequential()
model.add(Input(shape=(1,)))
model.add(Dense(3, activation='relu'))
model.add(Dense(1, activation='linear'))

# Step 2: Compile
model.compile(loss='mse' , metrics = ['mae'])

# Step 3: Fit
# 50 epochs means 50 complete passes of the training algorithm
# through the entire training dataset
model.fit(X_train_scaled, y_train,
          validation_data=(X_val_scaled, y_val),
          epochs=50)


Epoch 1/50
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 68708216832.0000 - mae: 233264.7344 - val_loss: 65196789760.0000 - val_mae: 220649.3438
Epoch 2/50
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 68708192256.0000 - mae: 233264.6719 - val_loss: 65196777472.0000 - val_mae: 220649.2812
Epoch 3/50
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 68708175872.0000 - mae: 233264.6562 - val_loss: 65196756992.0000 - val_mae: 220649.2500
Epoch 4/50
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 68708159488.0000 - mae: 233264.5781 - val_loss: 65196736512.0000 - val_mae: 220649.1719
Epoch 5/50
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 68708134912.0000 - mae: 233264.5625 - val_loss: 65196711936.0000 - val_mae: 220649.1406
Epoch 6/50
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 68708118528.0000 - mae: 233264.5156 - val_loss: 65196691456.0000 - val_mae: 220649.0781
Epoch 7/50
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 68708102144.0000 - mae: 233264.4844 - val_loss: 65196666880.000

In [31]:
# Model Weights
model.weights


[<Variable path=sequential_4/dense_9/kernel, shape=(1, 3), dtype=float32, value=[[ 1.3684869 -1.7924807 -2.0055506]]>,
 <Variable path=sequential_4/dense_9/bias, shape=(3,), dtype=float32, value=[0.86086124 0.7412715  0.86818796]>,
 <Variable path=sequential_4/dense_10/kernel, shape=(3, 1), dtype=float32, value=[[1.09295   ]
  [0.7768563 ]
  [0.98111814]]>,
 <Variable path=sequential_4/dense_10/bias, shape=(1,), dtype=float32, value=[0.8559053]>]

In [32]:
# Model Summary
model.summary()


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense_9 (Dense)                      │ (None, 3)                   │               6 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_10 (Dense)                     │ (None, 1)                   │               4 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 22 (92.00 B)

 Trainable params: 10 (40.00 B)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 12 (52.00 B)

## 10. Batch size (Chunks)

In [33]:
# Use the validation
# Step 1: Define
model = Sequential()
model.add(Input(shape=(1,)))
model.add(Dense(3, activation='relu'))
model.add(Dense(1, activation='linear'))

# Step 2: Compile
model.compile(loss='mse' , metrics = ['mae'])

# Step 3: Fit
model.fit(X_train_scaled, y_train,
          validation_data=(X_val_scaled, y_val),
          epochs=50,
          batch_size=64)


Epoch 1/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - loss: 68708118528.0000 - mae: 233264.3281 - val_loss: 65196654592.0000 - val_mae: 220648.7344
Epoch 2/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 68708110336.0000 - mae: 233264.2969 - val_loss: 65196642304.0000 - val_mae: 220648.7031
Epoch 3/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 68708085760.0000 - mae: 233264.2500 - val_loss: 65196630016.0000 - val_mae: 220648.6719
Epoch 4/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 68708077568.0000 - mae: 233264.2188 - val_loss: 65196613632.0000 - val_mae: 220648.6406
Epoch 5/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 68708073472.0000 - mae: 233264.2031 - val_loss: 65196605440.0000 - val_mae: 220648.5938
Epoch 6/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 68708057088.0000 - mae: 233264.1406 - val_loss: 65196584960.0000 - val_mae: 220648.5625
Epoch 7/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 68708032512.0000 - mae: 233264.1250 - val_loss: 65196580864.0000 - val_ma

In [34]:
# Model Weights
model.weights


[<Variable path=sequential_5/dense_11/kernel, shape=(1, 3), dtype=float32, value=[[ 0.5380359  -1.5803958  -0.45800984]]>,
 <Variable path=sequential_5/dense_11/bias, shape=(3,), dtype=float32, value=[0.324721   0.45979604 0.46495277]>,
 <Variable path=sequential_5/dense_12/kernel, shape=(3, 1), dtype=float32, value=[[0.361    ]
  [1.68041  ]
  [1.0434587]]>,
 <Variable path=sequential_5/dense_12/bias, shape=(1,), dtype=float32, value=[0.457996]>]

In [35]:
# Model Summary
model.summary()


Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense_11 (Dense)                     │ (None, 3)                   │               6 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_12 (Dense)                     │ (None, 1)                   │               4 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 22 (92.00 B)

 Trainable params: 10 (40.00 B)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 12 (52.00 B)

Try a different batch size. 

__Note:__ the best practice to have the value in powers of 2

In [36]:
# Use the validation
# Step 1: Define
model = Sequential()
model.add(Input(shape=(1,)))
model.add(Dense(3, activation='relu'))
model.add(Dense(1, activation='linear'))

# Step 2: Compile
model.compile(loss='mse' , metrics = ['mae'])

# Step 3: Fit
# try a different (power of 2) batch size
model.fit(X_train_scaled, y_train,
          validation_data=(X_val_scaled, y_val),
          epochs=50,
          batch_size=32)


Epoch 1/50
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - loss: 68708352000.0000 - mae: 233264.9375 - val_loss: 65196916736.0000 - val_mae: 220649.5469
Epoch 2/50
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 68708331520.0000 - mae: 233264.9219 - val_loss: 65196904448.0000 - val_mae: 220649.5156
Epoch 3/50
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 68708323328.0000 - mae: 233264.9062 - val_loss: 65196892160.0000 - val_mae: 220649.4844
Epoch 4/50
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 68708306944.0000 - mae: 233264.8438 - val_loss: 65196875776.0000 - val_mae: 220649.4688
Epoch 5/50
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 68708282368.0000 - mae: 233264.7969 - val_loss: 65196851200.0000 - val_mae: 220649.4219
Epoch 6/50
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 68708274176.0000 - mae: 233264.7812 - val_loss: 65196838912.0000 - val_mae: 220649.4062
Epoch 7/50
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 68708249600.0000 - mae: 233264.7656 - val_loss: 65196818432.0000

In [37]:
# Model Weights
model.weights


[<Variable path=sequential_6/dense_13/kernel, shape=(1, 3), dtype=float32, value=[[-0.8516436  1.618362  -1.4185404]]>,
 <Variable path=sequential_6/dense_13/bias, shape=(3,), dtype=float32, value=[0.35119838 0.51117486 0.86864465]>,
 <Variable path=sequential_6/dense_14/kernel, shape=(3, 1), dtype=float32, value=[[0.5751441 ]
  [0.60568917]
  [0.8501351 ]]>,
 <Variable path=sequential_6/dense_14/bias, shape=(1,), dtype=float32, value=[0.8549081]>]

In [38]:
# Model Summary
model.summary()


Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense_13 (Dense)                     │ (None, 3)                   │               6 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_14 (Dense)                     │ (None, 1)                   │               4 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 22 (92.00 B)

 Trainable params: 10 (40.00 B)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 12 (52.00 B)

## 11. History

**The Three Main Reasons to Use History**
  1. To Spot Overfitting
  2. To Visualise the Learning Curve
  3. To Know When to Stop Training

By saving the output of `model.fit()` to a variable (usually called `history`), Keras keeps a record of the training loss and validation loss at the end of each epoch. 

Plotting these two lines together helps us diagnose our model:
* If training loss goes down but validation loss goes up, the model is **overfitting** (memorizing the training data).
* If both are going down, the model is still learning.
* If both have flattened out, the model has converged and stopped learning.

In [39]:
# Use the validation
# Step 1: Define
model = Sequential()
model.add(Input(shape=(1,)))
model.add(Dense(3, activation='relu'))
model.add(Dense(1, activation='linear'))

# Step 2: Compile
model.compile(loss='mse' , metrics = ['mae'] , optimizer = Adam(learning_rate= 0.01))

# Step 3: Fit
history = model.fit(X_train_scaled, y_train,
                     validation_data=(X_val_scaled, y_val),
                     epochs=50,
                     batch_size=64)


NameError: name 'Adam' is not defined

In [ ]:
# Model Weights
model.weights


In [ ]:
# Model Summary
model.summary()


In [ ]:
# Create a History Data Frame
history_df = pd.DataFrame(history.history)
history_df


In [ ]:
# Visualize Epochs vs. Loss for the Training Loss and Validation Loss
history_df.plot()
plt.xlabel('epochs')
plt.ylabel('loss');


## 12. Early Stopping (callback)

- Prevent overfitting
- will not waste the computing power
- save time
  
It is hard to guess exactly how many epochs a model needs. If you train for too few epochs, the model underfits. If you train for too many, it overfits. 

An **Early Stopping Callback** acts as an automatic brake. It monitors the validation loss and automatically stops training once the validation score stops improving, even if you haven't reached your maximum number of epochs. It can also automatically restore the model's best weights!

In [ ]:
# configure EarlyStopping parameters
es = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)


In [ ]:
# Use the validation
# Step 1: Define
model = Sequential()
model.add(Input(shape=(1,)))
model.add(Dense(3, activation='relu'))
model.add(Dense(1, activation='linear'))

# Step 2: Compile
model.compile(loss='mse' , metrics = ['mae'] , optimizer = Adam(learning_rate= 0.01))

# Step 3: Fit
history = model.fit(X_train_scaled, y_train,
                     validation_data=(X_val_scaled, y_val),
                     epochs=100,
                     batch_size=64,
                     callbacks=[es])


In [ ]:
# Model Weights
model.weights


In [ ]:
# Model Summary
model.summary()


In [ ]:
# Create a History Data Frame
history_df = pd.DataFrame(history.history)
history_df


In [ ]:
# Visualize Epochs vs. Loss for the Training Loss and Validation Loss
history_df.plot()
plt.xlabel('epochs')
plt.ylabel('loss');


## 13.Multi-Feature Neural Network

In [ ]:
# configure EarlyStopping parameters
es = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# Use ALL the numeric features this time, not just sq__ft
X = housing[['beds', 'baths', 'sq__ft', 'latitude', 'longitude']]
y = housing['price']

# Split into Train / Validation / Test again with the new feature set
X_train_all, X_test, y_train_all, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_all, y_train_all, test_size=0.20, random_state=42)

# Scale the (now multi-column) features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)


In [ ]:
# Use the validation
# Step 1: Define
model = Sequential([
    Input(shape=(5,)),                     # 5 features now
    Dense(20, activation='relu'),
    Dense(15, activation='relu'),
    Dense(1, activation='linear')
])

# Step 2: Compile
model.compile(loss='mae')

# Step 3: Fit
history = model.fit(X_train_scaled, y_train,
                     validation_data=(X_val_scaled, y_val),
                     epochs=100,
                     batch_size=64,
                     callbacks=[es])


In [ ]:
# Model Weights
model.weights


In [ ]:
# Model Summary
model.summary()


In [ ]:
# Create a History Data Frame
history_df = pd.DataFrame(history.history)
history_df


In [ ]:
# Visualize Epochs vs. Loss for the Training Loss and Validation Loss
history_df.plot()
plt.xlabel('epochs')
plt.ylabel('loss');
